# BB84_Protocol

In [106]:
import numpy as np
import qiskit
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_aer import AerSimulator
import random

In [113]:
def qubits_from_source_to_destination(bit, basis_source, basis_destination, n_bits):

    bits = []
    simulator = AerSimulator()
    for i in range(n_bits):
        cr = ClassicalRegister(1)
        qr = QuantumRegister(1)
        qc = QuantumCircuit(qr, cr)
        if bit[i] == 1:
            qc.x(0)
        if basis_source[i] == 0: # '+' basis
            pass
        elif basis_source[i] == 1: # 'x' basis
            qc.h(0)

        if basis_destination[i] == 1: # 'x' basis
            qc.h(0)
        qc.measure(0, cr)

        result = simulator.run(qc, shots=1).result()
        bits.append(int(list(result.get_counts().keys())[0]))

    return bits

In [114]:
def BB84_protocol(alice_bits, alice_bases, eve_bases, bob_bases, n_bits):

    eve_bits = qubits_from_source_to_destination(alice_bits, alice_bases, eve_bases, n_bits)
    bob_bits_with_eve = qubits_from_source_to_destination(eve_bits, eve_bases, bob_bases, n_bits)
    bob_bits_without_eve = qubits_from_source_to_destination(alice_bits, alice_bases, bob_bases, n_bits)

    return bob_bits_with_eve, bob_bits_without_eve

In [121]:
n_bits = 1000

alice_bits = [random.randint(0, 1) for _ in range(n_bits)]
alice_bases = [random.randint(0, 1) for _ in range(n_bits)]
eve_bases = [random.randint(0, 1) for _ in range(n_bits)]
bob_bases = [random.randint(0, 1) for _ in range(n_bits)]

bob_bits_with_eve, bob_bits_without_eve = BB84_protocol(alice_bits, alice_bases, eve_bases, bob_bases, n_bits)


matched_bases = [i for i in range(n_bits) if alice_bases[i] == bob_bases[i]]
errors_with_eve = sum(1 for i in matched_bases if bob_bits_with_eve[i] != alice_bits[i])
errors_without_eve = sum(1 for i in matched_bases if bob_bits_without_eve[i] != alice_bits[i])

print(f"With Eve P(error): {errors_with_eve / len(matched_bases) * 100:.2f}%")
print(f"Without Eve P(error): {errors_without_eve / len(matched_bases) * 100:.2f}%")


With Eve P(error): 25.11%
Without Eve P(error): 0.00%
